# Updated Scraping Pipeline

This notebook rebuilds cleaner training pairs for pandas docs + Stack Overflow.

Key fixes:
- Parse **all** pandas doctest blocks instead of only the first block.
- Prefer doctest blocks that include the page API symbol.
- Prefer accepted or highest-score Stack Overflow answers with code.
- Apply quality filters before writing JSONL.

In [6]:
import json
import os
import re
import time
from pathlib import Path
from urllib.parse import urljoin, urlparse

import requests
from bs4 import BeautifulSoup
from tqdm import tqdm

DATA_ROOT = Path("./data_collection_2")
PANDAS_HTML_DIR = DATA_ROOT / "pandas_html"
PANDAS_JSONL = DATA_ROOT / "pandas.jsonl"
SO_JSON = DATA_ROOT / "stackoverflow" / "stackoverflow_data.json"
SO_JSONL = DATA_ROOT / "stackoverflow.jsonl"

for p in [PANDAS_HTML_DIR, SO_JSON.parent]:
    p.mkdir(parents=True, exist_ok=True)

In [7]:
# -------- Pandas docs scraping --------
PANDAS_INDEX = "https://pandas.pydata.org/docs/reference/index.html"
session = requests.Session()

resp = session.get(PANDAS_INDEX, timeout=30)
resp.raise_for_status()
index_soup = BeautifulSoup(resp.text, "html.parser")

# Step 1: collect reference section pages (io.html, frame.html, groupby.html, ...)
section_links = []
for a in index_soup.select(".toctree-l1 > a"):
    href = (a.get("href") or "").strip()
    if not href or href.startswith("http"):
        continue
    href = href.split("#")[0]
    if href.endswith(".html"):
        section_links.append(href)

section_links = sorted(set(section_links))
print("Reference sections:", len(section_links))

# Step 2: crawl each section page and collect API page links from tables
api_links = set()
for rel in tqdm(section_links):
    url = urljoin(PANDAS_INDEX, rel)
    try:
        section_resp = session.get(url, timeout=30)
        section_resp.raise_for_status()
    except Exception:
        continue

    section_soup = BeautifulSoup(section_resp.text, "html.parser")
    for a in section_soup.select("table a.internal"):
        href = (a.get("href") or "").strip()
        if not href or href.startswith("http"):
            continue
        href = href.split("#")[0]
        if href.startswith("api/") and href.endswith(".html"):
            api_links.add(href)

api_links = sorted(api_links)
print("Unique pandas API pages:", len(api_links))

Reference sections: 17


100%|██████████| 17/17 [00:02<00:00,  7.02it/s]

Unique pandas API pages: 2121


In [8]:
# Download pandas API pages locally
for rel in tqdm(api_links):
    url = urljoin(PANDAS_INDEX, rel)
    out = PANDAS_HTML_DIR / Path(rel).name
    if out.exists():
        continue
    try:
        r = session.get(url, timeout=30)
        r.raise_for_status()
        out.write_text(r.text, encoding="utf-8")
        time.sleep(0.03)
    except Exception:
        continue

print("Downloaded html files:", len(list(PANDAS_HTML_DIR.glob("*.html"))))

100%|██████████| 2121/2121 [03:24<00:00, 10.37it/s]

Downloaded html files: 2121


In [11]:
def infer_api_symbol_from_filename(filename: str) -> str:
    base = filename.replace(".html", "")
    if "." in base:
        return base.split(".")[-1]
    return ""


def extract_prompt(soup: BeautifulSoup) -> str:
    # Prefer API summary in definition list; fallback to first paragraph.
    q = soup.select_one("dd p")
    if q and q.get_text(strip=True):
        return q.get_text(" ", strip=True)
    p = soup.select_one("section p")
    return p.get_text(" ", strip=True) if p else ""


def extract_doctest_blocks(soup: BeautifulSoup):
    blocks = []
    for pre in soup.select("div.doctest pre"):
        lines = []
        # Use empty separator so syntax-highlighted spans remain on one code line.
        for raw in pre.get_text("", strip=False).splitlines():
            line = raw.rstrip()
            stripped = line.lstrip()
            if stripped.startswith(">>>"):
                code_line = stripped.split(">>>", 1)[1].strip()
                if code_line:
                    lines.append(code_line)
            elif stripped.startswith("..."):
                code_line = stripped.split("...", 1)[1].strip()
                if code_line:
                    lines.append(code_line)
        if lines:
            blocks.append("\n".join(lines))
    return blocks


def choose_answer_blocks(blocks, api_symbol: str):
    if not blocks:
        return ""

    if api_symbol:
        pat = re.compile(rf"(?:\.|\b){re.escape(api_symbol)}\s*\(")
        matching = [b for b in blocks if pat.search(b)]
        if matching:
            blocks = matching

    # Keep strongest 1-2 blocks by amount of executable content.
    blocks = sorted(blocks, key=lambda x: (x.count("\n"), len(x)), reverse=True)
    return "\n\n".join(blocks[:2]).strip()


def pandas_quality_filter(q: str, a: str) -> bool:
    if not q or not a:
        return False
    if len(q) < 12 or len(a) < 25:
        return False
    if "(" not in a:
        return False
    return True


pandas_data = []
for html_path in tqdm(sorted(PANDAS_HTML_DIR.glob("*.html"))):
    soup = BeautifulSoup(html_path.read_text(encoding="utf-8", errors="ignore"), "html.parser")
    q = extract_prompt(soup)
    blocks = extract_doctest_blocks(soup)
    api_symbol = infer_api_symbol_from_filename(html_path.name)
    a = choose_answer_blocks(blocks, api_symbol)
    if pandas_quality_filter(q, a):
        pandas_data.append({"q": q, "a": a})

print("Pandas pairs:", len(pandas_data))
print(pandas_data[0] if pandas_data else "No rows")

100%|██████████| 2121/2121 [01:45<00:00, 20.14it/s]

Pandas pairs: 2007
{'q': 'An ExtensionDtype for PyArrow data types.', 'a': 'pd.ArrowDtype(pa.timestamp("s", tz="America/New_York"))\npd.ArrowDtype(pa.list_(pa.int64()))\n\nimport pyarrow as pa\npd.ArrowDtype(pa.int64())'}


In [12]:
with PANDAS_JSONL.open("w", encoding="utf-8") as f:
    for item in pandas_data:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("Wrote", PANDAS_JSONL)

Wrote data_collection_2/pandas.jsonl


In [19]:
# -------- Stack Overflow scraping --------
BASE_URL = "https://api.stackexchange.com/2.3"
SITE = "stackoverflow"
TAGS = "python;pandas"
PAGESIZE = 100
TARGET_COUNT = 2500
MAX_PAGES = 25  # Stack Exchange search endpoints commonly cap pagination.


def _request_json(url: str, params: dict):
    r = requests.get(url, params=params, timeout=30)
    try:
        payload = r.json()
    except ValueError:
        payload = {"error_message": r.text[:300]}

    if r.status_code >= 400:
        msg = payload.get("error_message", f"HTTP {r.status_code}")
        raise RuntimeError(f"API request failed ({r.status_code}): {msg}")

    return payload


def fetch_questions(page: int):
    url = f"{BASE_URL}/search/advanced"
    params = {
        "page": page,
        "pagesize": PAGESIZE,
        "order": "desc",
        "sort": "creation",
        "tagged": TAGS,
        "answers": 1,
        "site": SITE,
        "filter": "withbody",
    }
    return _request_json(url, params)


def fetch_answers(question_ids):
    if not question_ids:
        return {"items": []}
    ids = ";".join(map(str, question_ids))
    url = f"{BASE_URL}/questions/{ids}/answers"
    params = {
        "order": "desc",
        "sort": "votes",
        "site": SITE,
        "filter": "withbody",
    }
    return _request_json(url, params)


def extract_question_text(title_html: str, body_html: str) -> str:
    q = title_html.strip() + "\n\n"
    soup = BeautifulSoup(body_html or "", "html.parser")
    for p in soup.select("p"):
        txt = p.get_text(" ", strip=True)
        if txt:
            q += txt + "\n"
    return q.strip()


def first_code_block(answer_html: str) -> str:
    soup = BeautifulSoup(answer_html or "", "html.parser")
    for code in soup.select("pre > code"):
        txt = code.get_text("\n", strip=True)
        if txt and "\n" in txt and len(txt) >= 25:
            return txt
    return ""


def build_stackoverflow_dataset(target_count=TARGET_COUNT):
    all_rows = []
    page = 1

    while len(all_rows) < target_count and page <= MAX_PAGES:
        print(f"\rFetching page {page}", end="", flush=True)
        try:
            q_data = fetch_questions(page)
        except RuntimeError as e:
            print(f"\nStopping early: {e}")
            break

        questions = q_data.get("items", [])
        if not questions:
            break

        qids = [q["question_id"] for q in questions]
        try:
            a_data = fetch_answers(qids)
        except RuntimeError as e:
            print(f"\nAnswer fetch failed on page {page}: {e}")
            a_data = {"items": []}

        answers_map = {}
        for ans in a_data.get("items", []):
            answers_map.setdefault(ans["question_id"], []).append(ans)

        for q in questions:
            qid = q["question_id"]
            q_text = extract_question_text(q.get("title", ""), q.get("body", ""))
            answers = answers_map.get(qid, [])

            # Prefer accepted answer first, then highest-vote with usable code.
            answers = sorted(answers, key=lambda x: (not x.get("is_accepted", False), -x.get("score", 0)))
            a_code = ""
            for ans in answers:
                a_code = first_code_block(ans.get("body", ""))
                if a_code:
                    break

            if q_text and a_code:
                all_rows.append({"q": q_text, "a": a_code})
                if len(all_rows) >= target_count:
                    break

        if q_data.get("has_more") is False:
            break

        if q_data.get("backoff"):
            time.sleep(q_data["backoff"])
        else:
            time.sleep(0.6)

        page += 1

    print()
    return all_rows


# Uncomment to fetch fresh data from Stack Exchange API:
so_data = build_stackoverflow_dataset()
with SO_JSON.open("w", encoding="utf-8") as f:
    json.dump(so_data, f, ensure_ascii=False, indent=2)
print("Fetched SO rows:", len(so_data))
print("Wrote", SO_JSON)

Fetching page 1

Fetching page 25
Fetched SO rows: 565
Wrote data_collection_2/stackoverflow/stackoverflow_data.json


In [20]:
# Convert fetched Stack Overflow data into final JSONL.
if SO_JSON.exists():
    raw_rows = json.loads(SO_JSON.read_text(encoding="utf-8"))
else:
    raw_rows = []


def first_code_from_html_answers(answers):
    for html in answers or []:
        soup = BeautifulSoup(html or "", "html.parser")
        for code in soup.select("pre > code"):
            txt = code.get_text("\n", strip=True)
            if txt and len(txt) >= 25:
                return txt
    return ""


# Normalize rows so this cell works with either schema:
# 1) [{'q':..., 'a':...}] OR 2) [{'title':..., 'question':..., 'answers':[html,...]}]
so_rows = []
for item in raw_rows:
    if "q" in item and "a" in item:
        q = item.get("q", "").strip()
        a = item.get("a", "").strip()
    else:
        title = item.get("title", "").strip()
        body_html = item.get("question", "")
        q = title + "\n\n"
        soup = BeautifulSoup(body_html or "", "html.parser")
        for p in soup.select("p"):
            txt = p.get_text(" ", strip=True)
            if txt:
                q += txt + "\n"
        q = q.strip()
        a = first_code_from_html_answers(item.get("answers", []))

    if q and a:
        so_rows.append({"q": q, "a": a})


def so_quality_filter(item):
    q, a = item.get("q", ""), item.get("a", "")
    if len(q) < 30 or len(a) < 25:
        return False
    # Keep either callable code or assignment-heavy snippets.
    if "(" not in a and "=" not in a:
        return False
    return True


clean_so = [x for x in so_rows if so_quality_filter(x)]
with SO_JSONL.open("w", encoding="utf-8") as f:
    for item in clean_so:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("Raw rows loaded:", len(raw_rows))
print("Normalized rows:", len(so_rows))
print("SO rows after filter:", len(clean_so))
print("Wrote", SO_JSONL)

Raw rows loaded: 565
Normalized rows: 565
SO rows after filter: 502
Wrote data_collection_2/stackoverflow.jsonl
